## Notebook overview

This notebook introduces three supervised learning models using scikit-learn:

1. A **Decision Tree** as a simple, interpretable baseline  
2. A **Random Forest** as an example of bagging-based ensemble learning  
3. **Gradient Boosting** as an example of boosting-based ensemble learning  

The dataset is stored in `vs_bank_part.csv`.

You will:
- Load the dataset from a CSV file
- Split the data using the existing `partition_Indicator` column
- Predict the binary target variable `b_tgt`
- Use two categorical predictors:
  - `cat_input1`
  - `cat_input2`
- Use twelve numeric predictors:
  - `logi_rfm1` to `logi_rfm12`
- Fit and compare three models
- Print the confusion matrix and multiple performance metrics for each model

## Dataset instructions

The file `vs_bank_part.csv` should be downloaded from Moodle and then uploaded into the Colab session.

In Google Colab:
1. Open the notebook
2. Click the folder icon on the left panel
3. Click **Upload**
4. Upload `vs_bank_part.csv`

After uploading, the file can be loaded using:

```python
df = pd.read_csv("vs_bank_part.csv")
```

Note that uploaded files are temporary in Colab and need to be uploaded again if the session resets.

## Imports

The next cell imports the Python tools needed for this notebook.

These imports include:
- **Pandas** and **NumPy** for working with data
- **ColumnTransformer**, **OneHotEncoder**, and **Pipeline** for preprocessing and modelling workflows
- **DecisionTreeClassifier**, **RandomForestClassifier**, and **GradientBoostingClassifier** for the three models
- performance metrics for evaluating classification results

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Evaluation
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score


## Load the dataset

We first load the CSV file and inspect its structure.

This helps confirm:
- the file loaded correctly
- the column names are available
- the partition column and target column exist

In [ ]:
# Load the dataset
df = pd.read_csv("vs_bank_part.csv")

# Basic inspection
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nFirst 5 rows:\n", df.head())


## Define target, predictors, and partition

The target variable is `b_tgt`.

The predictors include:
- two categorical variables: `cat_input1`, `cat_input2`
- twelve numeric variables: `logi_rfm1` to `logi_rfm12`

The dataset already contains a partition column called `partition_Indicator`, which will be used to create the training and validation datasets.

In [ ]:
# Target variable
target = "b_tgt"

# Categorical predictors
categorical_features = ["cat_input1", "cat_input2"]

# Numeric predictors
numeric_features = [
    "logi_rfm1", "logi_rfm2", "logi_rfm3", "logi_rfm4",
    "logi_rfm5", "logi_rfm6", "logi_rfm7", "logi_rfm8",
    "logi_rfm9", "logi_rfm10", "logi_rfm11", "logi_rfm12"
]

# All predictors together
all_features = categorical_features + numeric_features

print("Number of predictors:", len(all_features))
print("Predictors:", all_features)


## Create training and validation sets

Instead of using `train_test_split`, this dataset already provides a partition column.

We will:
- use rows where `partition_Indicator == "Training"` for model fitting
- use rows where `partition_Indicator == "Validation"` for evaluation

This is common in pre-partitioned business datasets.

In [ ]:
# Create training and validation subsets
train_df = df[df["partition_Indicator"] == "Training"].copy()
valid_df = df[df["partition_Indicator"] == "Validation"].copy()

print("Training shape:", train_df.shape)
print("Validation shape:", valid_df.shape)

# Split into X and y
X_train = train_df[all_features]
y_train = train_df[target]

X_valid = valid_df[all_features]
y_valid = valid_df[target]


## Handle categorical variables

Scikit-learn models require numeric input. Since `cat_input1` and `cat_input2` are categorical, we will convert them into dummy variables using **one-hot encoding**.

We will use a `ColumnTransformer` so that:
- categorical variables are one-hot encoded
- numeric variables are passed through unchanged

This preprocessing will be included inside each modelling pipeline.

In [ ]:
# Preprocessing:
# - one-hot encode categorical variables
# - keep numeric variables unchanged
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)


## Model 1: Decision Tree

A Decision Tree is a simple and interpretable model.

It works by repeatedly splitting the data into smaller groups based on predictor values.  
This makes it useful as a first baseline model.

In [ ]:
# Build a pipeline: preprocessing + decision tree
tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42, max_depth=4))
])

# Fit the model
tree_model.fit(X_train, y_train)

# Predict on validation data
tree_pred = tree_model.predict(X_valid)

# Evaluate
print("Decision Tree Accuracy:", accuracy_score(y_valid, tree_pred))
print("\nDecision Tree Confusion Matrix:\n", confusion_matrix(y_valid, tree_pred))


## Model 2: Random Forest

A Random Forest is an ensemble of many decision trees.

Instead of relying on a single tree, it builds many trees on different bootstrap samples and combines their predictions.  
This often improves predictive performance and reduces overfitting.

Parameters used here:
- `n_estimators=100`: builds 100 trees in the forest
- `max_depth=6`: limits tree depth to control complexity
- `max_features="sqrt"`: each split only considers a subset of predictors, which helps make trees more diverse
- `random_state=42`: makes results reproducible

In [ ]:
# Build a pipeline: preprocessing + random forest
forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=6,
        max_features="sqrt",
        random_state=42
    ))
])

# Fit the model
forest_model.fit(X_train, y_train)

# Predict on validation data
forest_pred = forest_model.predict(X_valid)

# Evaluate
print("Random Forest Accuracy:", accuracy_score(y_valid, forest_pred))
print("\nRandom Forest Confusion Matrix:\n", confusion_matrix(y_valid, forest_pred))


## Model 3: Gradient Boosting

Gradient Boosting is another ensemble method, but it works differently from Random Forest.

Instead of building trees independently, it builds trees sequentially.  
Each new tree focuses on correcting the errors made by the previous trees.

This often produces strong predictive performance, especially on structured tabular data.

Parameters used here:
- `n_estimators=100`: builds 100 boosting stages
- `learning_rate=0.1`: controls how much each new tree contributes
- `max_depth=3`: keeps each individual tree small and simple
- `max_features="sqrt"`: uses only a subset of predictors when searching for splits, which can improve generalisation
- `random_state=42`: makes results reproducible

In [ ]:
# Build a pipeline: preprocessing + gradient boosting
gb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        max_features="sqrt",
        random_state=42
    ))
])

# Fit the model
gb_model.fit(X_train, y_train)

# Predict on validation data
gb_pred = gb_model.predict(X_valid)

# Evaluate
print("Gradient Boosting Accuracy:", accuracy_score(y_valid, gb_pred))
print("\nGradient Boosting Confusion Matrix:\n", confusion_matrix(y_valid, gb_pred))


## Report multiple classification metrics

Accuracy is useful, but it does not tell the full story.

For binary classification, we often also report:
- **True Positive Rate (TP rate)**, also called **Sensitivity** or **Recall**
- **True Negative Rate (TN rate)**, also called **Specificity**
- **F1 score**, which balances precision and recall

The next cell defines a small helper function to calculate and print these metrics from the confusion matrix.

In [ ]:
def print_metrics(model_name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)

    # For binary classification:
    # cm = [[TN, FP],
    #       [FN, TP]]
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    tp_rate = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    tn_rate = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    sensitivity = tp_rate
    specificity = tn_rate
    f1 = f1_score(y_true, y_pred)

    print(f"\n{model_name}")
    print("-" * len(model_name))
    print("Confusion Matrix:\n", cm)
    print("Accuracy:", round(accuracy, 4))
    print("TP rate:", round(tp_rate, 4))
    print("TN rate:", round(tn_rate, 4))
    print("Sensitivity:", round(sensitivity, 4))
    print("Specificity:", round(specificity, 4))
    print("F1 score:", round(f1, 4))

# Print full metrics for each model
print_metrics("Decision Tree", y_valid, tree_pred)
print_metrics("Random Forest", y_valid, forest_pred)
print_metrics("Gradient Boosting", y_valid, gb_pred)


## Compare model performance in a table

We now compare the three models side by side using a summary table.

This makes it easier to see whether the ensemble models improved on the single decision tree baseline.

In [ ]:
def get_metrics_dict(model_name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    tp_rate = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    tn_rate = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    sensitivity = tp_rate
    specificity = tn_rate
    f1 = f1_score(y_true, y_pred)

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "TP rate": tp_rate,
        "TN rate": tn_rate,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "F1 score": f1
    }

results = pd.DataFrame([
    get_metrics_dict("Decision Tree", y_valid, tree_pred),
    get_metrics_dict("Random Forest", y_valid, forest_pred),
    get_metrics_dict("Gradient Boosting", y_valid, gb_pred)
])

print(results.sort_values("Accuracy", ascending=False))


## Interpretation

Points to discuss after running the notebook:

- Did Random Forest improve on the single Decision Tree?
- Did Gradient Boosting perform better than both?
- Were the improvements small or large?
- Did all metrics improve, or only some of them?
- Which model would you prefer if interpretability mattered?
- Which model would you prefer if predictive accuracy mattered most?

## Optional extension ideas

Possible extensions for students:

- Change tree depth and compare results
- Change the number of trees in the Random Forest
- Change the learning rate in Gradient Boosting
- Change `max_features` and observe the effect
- Print predicted probabilities instead of only class labels
- Compute additional metrics such as precision and recall separately